In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.interpolate import interpn

from thermal_loads_260924 import *

In [ ]:
# environment settings: 
pd.set_option('display.max_column',None)
pd.set_option('display.max_rows',50)
pd.set_option('display.max_seq_items',None)
pd.set_option('display.max_colwidth', 500)
pd.set_option('expand_frame_repr', True)

np._core.arrayprint.set_printoptions(linewidth= 180)
np.set_printoptions(threshold=np.inf)

# Data

In [ ]:
project = '260924_House'

In [ ]:
df_gen_data = pd.read_excel(project + '.xlsx', sheet_name="general_data")
gen_data    = df_gen_data['Values']
df_gen_data = df_gen_data.set_index('Names')
df_gen_data

#### Additional Power for Reheating

In [ ]:
df_rh_data  = pd.read_excel(project + '.xlsx', sheet_name="reheat_data")
rh_data     = df_rh_data['Values']
df_rh_data = df_rh_data.set_index('Names')
df_rh_data

#### Ventilation

In [ ]:
dfv = pd.read_excel(project + '.xlsx', sheet_name="ventilation_data")
dfv = dfv.set_index('zone')
dfv

#### Materials

In [ ]:
materials = pd.read_excel(project + '.xlsx', sheet_name="materials_data")
materials = materials.set_index('material')
materials.round(2)

#### Wall types layers

In [ ]:
wt_layers = pd.read_excel(project + '.xlsx', sheet_name="wall_types_layers_data")
wt_layers = wt_layers.set_index('wall_type')
wt_layers

#### Wall types environments

In [ ]:
wt_env = pd.read_excel(project + '.xlsx', sheet_name="wall_types_env_data")
wt_env = wt_env.set_index('wall_type')
wt_env

#### Wall dimensions and environments

In [ ]:
wl = pd.read_excel(project + '.xlsx', sheet_name="wall_data")
wl = wl.set_index('zone')
wl

#### Windows

In [ ]:
wd = pd.read_excel(project + '.xlsx', sheet_name="windows_data")
wd = wd.set_index('zone')
wd

#### Doors

In [ ]:
dr = pd.read_excel(project + '.xlsx', sheet_name="doors_data")
dr = dr.set_index('zone')
dr

# Compute Heat Losses

In [ ]:
# dfwt  : Walls thicknesses [m] and conductive resistance [m²K/W]
# dfAU  : Walls U_values [W/m²K] - Temperature Factors - Areas [m²] - Heat Tranfer Coefficients [W/K]
# dfgrAU : Walls Areas [m²] and Heat Tranfer Coefficients [W/K] by zone and wall type
# dfAHT : Wall Areas [m²] and Heat Tranfer Coefficients [W/K] by zone
# dfH   : Total Heat Tranfer Coefficients by zone [W/K]
# dfPHI : Emission and production Heat losses and Loads by zone [W]
# dffv  : Emission Heat losses and loads by zone and by facade [W]
# dffPHI : Emission Heat losses and loads by facade [W]

dfwt, dfAU, dfgrAU, dfAHT, dfH, dfPHI, dffv, dffPHI = thermal_loads(gen_data, rh_data, dfv, \
                  materials, wt_layers, wt_env, wl, wd, dr)

In [ ]:
# Display style for the dataframes
styles = [dict(selector="caption",
                       props=[("text-align", "middle"),
                              ("font-size", "110%"),
                              ("font-weight", "bold"),
                              ("color", 'black'),
                              ("padding-bottom", "10px"),
                              ("white-space", "nowrap")
                             ])]

In [ ]:
numeric_cols = dfAU.select_dtypes(include='number').columns

dfAU.style \
    .format("{:.2f}", subset=numeric_cols) \
    .set_caption("Walls U_values [W/m²K] - Temperature Factors - Areas [m²] - Heat Tranfer Coefficients [W/K]") \
    .set_table_styles(styles)

In [ ]:
numeric_cols = dfgrAU.select_dtypes(include='number').columns

dfgrAU.style \
    .format("{:.2f}", subset=numeric_cols) \
    .set_caption("Walls Areas [m²] and Heat Tranfer Coefficients [W/K] by zone and wall type") \
    .set_table_styles(styles)

In [ ]:
bold_cols = ['area', 'H_T']
bold_cols = [col for col in dfAHT.columns if col.startswith(tuple([x + ' ' for x in bold_cols]))]

dfAHT.style \
    .format("{:.2f}") \
    .set_properties(subset=bold_cols, **{"font-weight": "bold"}) \
    .set_caption("Wall Areas [m²] and Heat Tranfer Coefficients [W/K] by zone") \
    .set_table_styles(styles)

In [ ]:
bold_cols = ['H_T', 'H_V', 'H_TOT']
bold_cols = [col for col in dfH.columns if col.startswith(tuple([x + ' ' for x in bold_cols]))]

dfH.style \
    .format("{:.1f}") \
    .set_properties(subset=bold_cols, **{"font-weight": "bold"}) \
    .set_caption("Total Heat Tranfer Coefficients by zone [W/K]") \
    .set_table_styles(styles)

In [ ]:
bold_cols = ['PHI_T', 'PHI_V', 'PHI_TOT', 'PHI_HL']
bold_cols = [col for col in dfPHI.columns if col.startswith(tuple([x + ' ' for x in bold_cols]))]

dfPHI.style \
    .format("{:.0f}") \
    .set_properties(subset=bold_cols, **{"font-weight": "bold"}) \
    .set_caption("Emission and production Heat losses and Loads by zone [W]") \
    .set_table_styles(styles)

In [ ]:
dffv.style \
    .format("{:.0f}") \
    .set_caption("Emission Heat losses and loads by zone and by facade [W]") \
    .set_table_styles(styles)

In [ ]:
dffPHI.style \
    .format("{:.0f}") \
    .set_caption("Emission Heat losses and loads by facade [W]") \
    .set_table_styles(styles)

In [ ]:
lst_cols = ['PHI_T_wl', 'PHI_T_wd', 'PHI_T_dr', 'PHI_V_su', 'PHI_V_ie', 'PHI_RH']
cols = [col for col in dfPHI.columns if col.startswith(tuple([x + ' ' for x in lst_cols]))]
df_plot = dfPHI[cols].copy()
df_plot.index.name = ''

red       = (0.984313725490196, 0.5019607843137255, 0.4470588235294118) 
orange    = (0.9921568627450981, 0.7058823529411765, 0.3843137254901961)  
yellow    = (1.0, 0.9294117647058824, 0.43529411764705883)
blue      = (0.09019607843137255, 0.7450980392156863, 0.8117647058823529)
lightblue = (0.6196078431372549, 0.8549019607843137, 0.8980392156862745)  
grey      = (0.83, 0.83, 0.83) 

df_plot.loc[::-1].plot.barh(figsize=(8,3), stacked=True, color = (red, orange, yellow, blue, lightblue, grey))
plt.xlabel("Heating Power [W]")
plt.title('Breakdown of the zone Heat Loads [W]')
plt.grid(linestyle = 'dotted');

In [ ]:
# dfres.to_excel('House_heat_flows.xlsx')

In [ ]:
with pd.ExcelWriter(project + "_output.xlsx") as writer: 

    # DATA
    
    # General data
    df_gen_data.to_excel(writer, sheet_name='general_data')

    # Reheat data
    df_rh_data.to_excel(writer, sheet_name='reheat_data') 

    # Ventilation data
    dfv.to_excel(writer, sheet_name='ventilation_data') 

    # Materials data
    materials.to_excel(writer, sheet_name='materials_data')

    # Wall types layers data
    wt_layers.to_excel(writer, sheet_name='wall_types_layers_data')

    # Wall types environmental data
    wt_env.to_excel(writer, sheet_name='wall_types_env_data')

    # Walls data
    wl.to_excel(writer, sheet_name='wall_data')

    # Windows data
    wd.to_excel(writer, sheet_name='windows_data')

    # Doors data
    dr.to_excel(writer, sheet_name='doors_data')
    

    # RESULTS

    # dfwt : Walls thicknesses [m] and conductive resistance [m²K/W]
    dfwt.round(2).to_excel(writer, sheet_name='walls_R') 
    
    # dfAU  : Walls U_values [W/m²K] - Temperature Factors - Areas [m²] - Heat Tranfer Coefficients [W/K]
    dfAU.round(2).to_excel(writer, sheet_name='walls_AU') 

    # dfgrAU : Walls Areas [m²] and Heat Tranfer Coefficients [W/K] by zone and wall type
    dfgrAU.round(2).round(2).to_excel(writer, sheet_name='walls_AU_grouped') 

    # dfAHT : Wall Areas [m²] and Heat Tranfer Coefficients [W/K] by zone
    dfAHT.round(2).to_excel(writer, sheet_name='HT_by_zone') 

    # dfH   : Total Heat Tranfer Coefficients by zone [W/K]
    dfH.round(2).to_excel(writer, sheet_name='HTOT_by_zone') 

    # dfPHI : Emission and production Heat losses and Loads by zone [W]
    dfPHI.astype(int).to_excel(writer, sheet_name='HL_by_zone') 

    # dffv  : Emission Heat losses and loads by zone and by facade [W]
    dffv.astype(int).to_excel(writer, sheet_name='HL_by_zone_by_facade') 

    # dffPHI : Emission Heat losses and loads by facade [W]
    dffPHI.astype(int).to_excel(writer, sheet_name='HL_by_facade') 